In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve
#sobel edge detection
gray_img = Image.open("gray_output.jpg").convert("L")
gray = np.array(gray_img, dtype=np.float32)

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],dtype=np.float32)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],dtype=np.float32)

grad_x = convolve(gray, sobel_x)
grad_y = convolve(gray, sobel_y)

grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)
grad_magnitude = np.clip(grad_magnitude,0,255).astype(np.uint8)

Image.fromarray(grad_magnitude).save("edge_sobel.jpg")

#laplacian edge detection
kernel = np.array([[0.5, 1.5, 0.5], [1.5, -8, 1.5], [0.5, 1.5, 0.5]], dtype=np.float32)
egde_laplacian = convolve(gray, kernel)
egde_laplacian = np.clip(egde_laplacian, 0, 255).astype(np.uint8)

Image.fromarray(egde_laplacian).save("edge_laplacian.jpg")

#FFT
fft_result = np.fft.fft2(gray)
fft_shifted = np.fft.fftshift(fft_result)

h,w = gray.shape
center_x, center_y = w//2, h//2
radius = int(min(center_x, center_y) * 0.15)  # 15% of the smaller dimension
#radius = 30
mask = np.zeros((h,w), dtype=np.uint8)
y,x = np.ogrid[:h, :w]
mask[(x-center_x)**2 + (y-center_y)**2 > radius**2] = 1

fft_shifted_filtered = fft_shifted * mask

fft_filtered = np.fft.ifftshift(fft_shifted_filtered)
edge_fft = np.fft.ifft2(fft_filtered)
edge_fft_magnitude = np.abs(edge_fft)

edge_fft_magnitude = np.clip(edge_fft_magnitude*3, 0, 255).astype(np.uint8)
Image.fromarray(edge_fft_magnitude).save("edge_fft.jpg")